# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    B, C, H, W = x.shape
    P = patch_size
    x = x.unfold(2, P, P).unfold(3, P, P)
    x = x.permute(0, 2, 3, 1, 4, 5)
    Hg, Wg = x.shape[1], x.shape[2]
    x = x.reshape(B, Hg * Wg, C * P * P)
    return x

x = torch.randn(2, 1, 28, 28)
patches = patchify(x, patch_size=4)
print(patches.shape)

torch.Size([2, 49, 16])


In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        self.proj = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        return self.proj(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos_embed

pe = PositionalEmbedding(49, 64)
tokens = torch.zeros(2, 49, 64)
print(pe(tokens).shape)

torch.Size([2, 49, 64])


In [ ]:
# shazeer just does some studies on performance and says that GEGLU and SwiGLU show better performance
# GEGLU and SwiGLU can amplify and not just fit the inputs into (0,1) and it also can go a little negative. 
#there difference is that GELU is defined by a Gaussian CDF and SwiGLU is defined by x * sigmoid(x). so this are different activation functions that bound in a very similar way
#shzeer thou does not give a clear reason why these perform better

class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class GLUFeedForward(nn.Module):
    """GLU-family FFN (GEGLU or SwiGLU)"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        self.variant = variant
        self.fc_gate = nn.Linear(d_model, d_ff_gated)
        self.fc_val = nn.Linear(d_model, d_ff_gated)
        self.fc_out = nn.Linear(d_ff_gated, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = self.fc_gate(x)
        val = self.fc_val(x)
        if self.variant == "geglu":
            gate = F.gelu(gate)
        elif self.variant == "swiglu":
            gate = F.silu(gate)
        else:
            raise ValueError(f"Unknown variant: {self.variant}")
        return self.drop(self.fc_out(gate * val))

ffn = FeedForward(64, 256, 0.1)
glu = GLUFeedForward(64, int(2 * 256 / 3), 0.1, "geglu")
x = torch.randn(2, 49, 64)
print(ffn(x).shape, glu(x).shape)

torch.Size([2, 49, 64]) torch.Size([2, 49, 64])


In [5]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.mlp = mlp
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN self-attention
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + self.drop(attn_out)
        # Pre-LN MLP
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

block = TransformerEncoderBlock(64, 4, FeedForward(64, 256, 0.1), dropout=0.1)
x = torch.randn(2, 49, 64)
print(block(x).shape)

torch.Size([2, 49, 64])


In [ ]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size  

        self.patch_embed = PatchEmbed(patch_dim, d_model)
        self.pos_embed = PositionalEmbedding(self.num_tokens, d_model)

        d_ff_gated = int(2 * d_ff / 3)

        def make_mlp():
            if mlp_kind == "ffn":
                return FeedForward(d_model, d_ff, dropout)
            else:
                return GLUFeedForward(d_model, d_ff_gated, dropout, mlp_kind)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=make_mlp(),
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        patches = patchify(x, self.patch_size)    
        tokens = self.patch_embed(patches)       
        tokens = self.pos_embed(tokens)
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)
        pooled = tokens.mean(dim=1)              
        logits = self.head(pooled)                  
        return logits

vit = TinyViT(patch_size=4, d_model=64, n_heads=4, n_layers=2, d_ff=256, dropout=0.1, mlp_kind="ffn")
x = torch.randn(2, 1, 28, 28)
print(vit(x).shape)
print(f"Params: {sum(p.numel() for p in vit.parameters()):,}")

torch.Size([2, 10])
Params: 104,970


In [7]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [8]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")

    return {
        "mlp_kind": mlp_kind,
        "train_losses": train_losses,
        "test_accs": test_accs,
        "final_test_acc": test_accs[-1],
        "best_test_acc": max(test_accs),
        "num_params": sum(p.numel() for p in model.parameters()),
    }

In [9]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="cpu")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model config
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

# Run ablation: baseline FFN vs GEGLU vs SwiGLU
runs = ["ffn", "geglu", "swiglu"]
results = []

for kind in runs:
    torch.manual_seed(cfg.seed)
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\nRun: {kind} | Params: {num_params:,}")
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)

print("\n=== Summary ===")
for r in results:
    print(f"{r['mlp_kind']:10s} | params: {r['num_params']:,} | "
          f"best acc: {r['best_test_acc']:.4f} | final acc: {r['final_test_acc']:.4f}")


Run: ffn | Params: 104,970
[ffn] epoch 1/5 | test acc: 0.8488
[ffn] epoch 2/5 | test acc: 0.9116
[ffn] epoch 3/5 | test acc: 0.9209
[ffn] epoch 4/5 | test acc: 0.9367
[ffn] epoch 5/5 | test acc: 0.9448

Run: geglu | Params: 104,882
[geglu] epoch 1/5 | test acc: 0.8734
[geglu] epoch 2/5 | test acc: 0.9338
[geglu] epoch 3/5 | test acc: 0.9550
[geglu] epoch 4/5 | test acc: 0.9547
[geglu] epoch 5/5 | test acc: 0.9631

Run: swiglu | Params: 104,882
[swiglu] epoch 1/5 | test acc: 0.8774
[swiglu] epoch 2/5 | test acc: 0.9344
[swiglu] epoch 3/5 | test acc: 0.9546
[swiglu] epoch 4/5 | test acc: 0.9545
[swiglu] epoch 5/5 | test acc: 0.9589

=== Summary ===
ffn        | params: 104,970 | best acc: 0.9448 | final acc: 0.9448
geglu      | params: 104,882 | best acc: 0.9631 | final acc: 0.9631
swiglu     | params: 104,882 | best acc: 0.9589 | final acc: 0.9589
